# 0. [OPTIONAL] Installing course dependencies

In [1]:
# run the script `init_uv.sh` before running this notebook.
# `sh init_uv.sh`
!cat init_uv.sh

#!/bin/sh

# Initialize the repository
uv init -q --description "Information Retrieval Workspace" . 

# Create a virtual environment
uv venv .venv

# Activate the venv
# . .venv/bin/activate

In [2]:
# show version of uv
!uv -V

uv 0.12.8 (x86_64-unknown-linux-gnu)


In [3]:
# upgrade uv
!uv self update

info: Checking for updates...
success: You're already on version v0.12.8 of uv (the latest version).


In [4]:
# install dependencies
!uv add requests
!uv add bs4
!uv add feedparser
!uv add nltk

Resolved 77 packages in 11ms
Checked 48 packages in 256ms
Resolved 77 packages in 9ms
Checked 48 packages in 295ms
Resolved 77 packages in 9ms
Checked 48 packages in 242ms
Resolved 77 packages in 10ms
Checked 48 packages in 284ms


In [5]:
!uv tree -d 1

Resolved 77 packages in 3ms
ir v0.1.0
├── bs4 v0.0.2
├── feedparser v6.0.14
├── ipykernel v7.3.0
├── nltk v3.10.3
└── requests v2.34.2


# 1. Touching the Internet

## 1.1. Intro

Solve the following task.
1. Download [this page](https://raw.githubusercontent.com/IUCVLab/information-retrieval/main/datasets/facts.txt)
2. Save it to the file with the **unique** name derived from the URL. 
3. Save another file with another URL and do not save it into the file with the previous name. E.g. [this file](https://github.com/IUCVLab/information-retrieval/blob/main/datasets/facts.txt) is another file with another content!

Hints:
- [requests](https://docs.python-requests.org/en/latest/) library is cool.
- [hashlib](https://docs.python.org/3/library/hashlib.html) may help with computing hash strings.
- when you download and save the data, don't try to encode and decode it. Use binary format when working with streams and files.


Some notes on unique file name from URLs:
- Please, never try to convert a domain (`google.com`), or a path component (`/index.php`) into a filename. They are not unique!
- Also, better not to try to substitute sensitive symbols of the full URL (`/:`...) -- you will definitely forget one. Also, you may easily overflow file name.
- Nice way is to use hash strings with fixed length and character set. Compute hash strings from the previous list.

In [1]:
import requests
from hashlib import sha512

url1 = "https://raw.githubusercontent.com/IUCVLab/information-retrieval/main/datasets/facts.txt"
url2 = "https://github.com/IUCVLab/information-retrieval/blob/main/datasets/facts.txt"
r1 = requests.get(url1)
r2 = requests.get(url2)

def download_and_save(url):
    response = requests.get(url)
    response.raise_for_status()
    
    # Create unique filename from URL
    url_hash = sha512(url.encode()).hexdigest()[:16]
    filename = f"download_{url_hash}.txt"
    
    # Save in binary mode
    with open(filename, 'wb') as f:
        f.write(response.content)
    
    return filename

# Download both files
filename1 = download_and_save(url1)
filename2 = download_and_save(url2)

print(f"Downloaded {url1} as {filename1}")
print(f"Downloaded {url2} as {filename2}")

# Verify they are different files
with open(filename1, 'rb') as f1, open(filename2, 'rb') as f2:
    content1 = f1.read()
    print(content1[:100])
    content2 = f2.read()
    print(content2[:100])
    print(content2[:100].decode().strip())

Downloaded https://raw.githubusercontent.com/IUCVLab/information-retrieval/main/datasets/facts.txt as download_cd4ee390e5f46f55.txt
Downloaded https://github.com/IUCVLab/information-retrieval/blob/main/datasets/facts.txt as download_1069a4438167ca55.txt
b'1. If you somehow found a way to extract all of the gold from the bubbling core of our lovely little'
b'\n\n\n\n  \n\n<!DOCTYPE html>\n<html\n  lang="en"\n  \n  data-color-mode="auto" data-light-theme="light" data-'
<!DOCTYPE html>
<html
  lang="en"
  
  data-color-mode="auto" data-light-theme="light" data-


[ASCII](https://en.wikipedia.org/wiki/ASCII) encodes each code-point as a value from 0 to 127 – storable as a seven-bit intege. Ninety-five code-points are printable, including digits 0 to 9, lowercase letters a to z, uppercase letters A to Z, and commonly used punctuation symbols. For example, the letter `i` is represented as 105 (decimal).

**ASCII** was the first character encoding standard for the web. It defined 128 different latin characters that could be used on the internet:
* English letters (a-z and A-Z)
* Numbers (0-9)
* Some special characters: ! $ + - ( ) @ < > . # ?


**ANSI (Windows-1252)** was the first Windows character set:
* Identical to ASCII for the first 127 characters
* Special characters from 128 to 159
* Identical to UTF-8 from 160 to 255

The **ISO-8859-1** Character Set was the default character set for HTML 4.
It supported 256 characters:
* Identical to ASCII for the first 127 characters
* Does not use the characters from 128 to 159
* Identical to ANSI and UTF-8 from 160 to 255


The **UTF-8** Character Set
* Identical to ASCII for the values from 0 to 127
* Does not use the characters from 128 to 159
* Identical to ANSI and 8859-1 from 160 to 255
* Continues from the value 256 to 10 000 characters

[Percent-encoding](https://en.wikipedia.org/wiki/Percent-encoding), also known as URL encoding, is a method to encode arbitrary data in a uniform resource identifier (URI) using only the ASCII characters legal within a URI. Percent-encoding is used to ensure special characters do not interfere with the URI's structure and interpretation. Special characters are replaced with a percent sign (%) followed by two hexadecimal digits representing the character's byte value. For example, a space is commonly encoded as `%20`:
* original: `http://example.com/my file.txt`
* encoded: `http://example.com/my%20file.txt`

## 1.2 Crawling
### A. Regular expressions

Regular expressions (aka regex, regexp) are used to search for patterns. Machine-readable languages often have regualar structure (not always), or at least are non-ambiguous.

Obvious way is, of course, to let machine parse the document and then process the result (as in the previous lab). But this often result in additinal depenencies and significant memory and time overhead (which is ok for a single document, but won't work for millions).


In [2]:
import re
string = "we have only 5 do11ars. This amount of $ is small. How should we sur-vive?"

# all alphanumerics
pattern = "\w+"
print(pattern, end=": ")
print(re.findall(pattern, string))
print()

# all alphanumerics but also with hyphen
pattern = "[\w\-]+"
print(pattern, end=": ")
print(re.findall(pattern, string))
print()

# the same but using explicit character enumeration
pattern = "[0-9a-zA-Z\-]+" # explicit
print(pattern, end=": ")
print(re.findall(pattern, string))
print()

pattern = ".+"
print(pattern, end=": ")
print(re.findall(pattern, string))
print()

# non-spaces, not the same as \w !
pattern = "\S+"
print(pattern, end=": ")
print(re.findall(pattern, string))
print()


# discuss this pattern
pattern = "\W+[a-z]+\-[a-z]+.$"
print(pattern, end=": ")
print(re.findall(pattern, string))

\w+: ['we', 'have', 'only', '5', 'do11ars', 'This', 'amount', 'of', 'is', 'small', 'How', 'should', 'we', 'sur', 'vive']

[\w\-]+: ['we', 'have', 'only', '5', 'do11ars', 'This', 'amount', 'of', 'is', 'small', 'How', 'should', 'we', 'sur-vive']

[0-9a-zA-Z\-]+: ['we', 'have', 'only', '5', 'do11ars', 'This', 'amount', 'of', 'is', 'small', 'How', 'should', 'we', 'sur-vive']

.+: ['we have only 5 do11ars. This amount of $ is small. How should we sur-vive?']

\S+: ['we', 'have', 'only', '5', 'do11ars.', 'This', 'amount', 'of', '$', 'is', 'small.', 'How', 'should', 'we', 'sur-vive?']

\W+[a-z]+\-[a-z]+.$: [' sur-vive?']


### B. Find URLs/URIs vs parse the doc

Instead of building DOM model and extracting `href` and `src` attributes, you may rely on the structure of the url itself. Extact all URLs from [the page](https://math.stackexchange.com/questions/411486/understanding-the-singular-value-decomposition-svd) with regexp. You major tool is [re.findall(...)](https://docs.python.org/3/library/re.html#). You may also be interested in compiled regular rexpression (if you reuse one).

In [3]:
import re
import requests

url = "https://math.stackexchange.com/questions/"\
        "411486/understanding-the-singular-value-decomposition-svd"

text = requests.get(url).text

# my inspiration - 
# I took some example URL regexp from the internet, 
# specifically from here:
# https://stackoverflow.com/questions/3809401/what-is-a-good-regular-expression-to-match-a-url
expressions = [
    "(?:([A-Za-z]+):)?(\/{0,3})([0-9.\-A-Za-z]+)(?::(\d+))?(?:\/([^?#]*))?(?:\?([^#]*))?(?:#(.*))?",
    "(www|http:|https:)+[^\s]+[\w]",
    "https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)",
    "[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)?",
    "(https?:\/\/(?:www\.|(?!www))[a-zA-Z0-9][a-zA-Z0-9-]+[a-zA-Z0-9]\.[^\s]{2,}|www\.[a-zA-Z0-9][a-zA-Z0-9-]+[a-zA-Z0-9]\.[^\s]{2,}|https?:\/\/(?:www\.|(?!www))[a-zA-Z0-9]+\.[^\s]{2,}|www\.[a-zA-Z0-9]+\.[^\s]{2,})",
    "(?!mailto:)(?:(?:http|https|ftp)://)(?:\\S+(?::\\S*)?@)?(?:(?:(?:[1-9]\\d?|1\\d\\d|2[01]\\d|22[0-3])(?:\\.(?:1?\\d{1,2}|2[0-4]\\d|25[0-5])){2}(?:\\.(?:[0-9]\\d?|1\\d\\d|2[0-4]\\d|25[0-4]))|(?:(?:[a-z\\u00a1-\\uffff0-9]+-?)*[a-z\\u00a1-\\uffff0-9]+)(?:\\.(?:[a-z\\u00a1-\\uffff0-9]+-?)*[a-z\\u00a1-\\uffff0-9]+)*(?:\\.(?:[a-z\\u00a1-\\uffff]{2,})))|localhost)(?::\\d{2,5})?(?:(/|\\?|#)[^\\s]*)?",
    "https?:\/\/(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&\/=]*)",
]

for expression in expressions:
    print()
    pattern = re.compile(expression)
    urls = pattern.findall(text)
    print(expression)
    print(urls[:10])


(?:([A-Za-z]+):)?(\/{0,3})([0-9.\-A-Za-z]+)(?::(\d+))?(?:\/([^?#]*))?(?:\?([^#]*))?(?:#(.*))?
[('', '', 'DOCTYPE', '', '', '', ''), ('', '', 'html', '', '', '', ''), ('', '', 'html', '', '', '', ''), ('', '', 'lang', '', '', '', ''), ('', '', 'en-US', '', '', '', ''), ('', '', 'head', '', '', '', ''), ('', '', 'title', '', '', '', ''), ('', '', 'Just', '', '', '', ''), ('', '', 'a', '', '', '', ''), ('', '', 'moment...', '', '', '', '')]

(www|http:|https:)+[^\s]+[\w]
['https:', 'https:', 'https:', 'https:', 'https:']

https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6([-a-zA-Z0-9()@:%_\+.~#?&//=]*)
[]

[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6([-a-zA-Z0-9()@:%_\+.~#?&//=]*)?
[]

(https?:\/\/(?:www\.|(?!www))[a-zA-Z0-9][a-zA-Z0-9-]+[a-zA-Z0-9]\.[^\s]{2,}|www\.[a-zA-Z0-9][a-zA-Z0-9-]+[a-zA-Z0-9]\.[^\s]{2,}|https?:\/\/(?:www\.|(?!www))[a-zA-Z0-9]+\.[^\s]{2,}|www\.[a-zA-Z0-9]+\.[^\s]{2,})
['https://challenges.cloudflare.com;', 'https://challenges.cloudflare.com;',

Was this success? 

Compose your own minimalistic:

In [4]:
import re
import requests

url = "https://math.stackexchange.com/questions/"\
        "411486/understanding-the-singular-value-decomposition-svd"

text = requests.get(url).text

protocol = "https?://"
domain = "[\w\-\.]+"
path = "[/\w\-\.]*"
args =  "(?:\?[\w\=&-_;\[\]]+)?"
hashtail = "(?:#[\w$%-_;]+)?"

expression = protocol + domain + path + args + hashtail
pattern = re.compile(expression)
regexp_urls = pattern.findall(text)
print(regexp_urls[:20])

['https://challenges.cloudflare.com', 'https://challenges.cloudflare.com', 'https://challenges.cloudflare.com', 'https://challenges.cloudflare.com', 'https://challenges.cloudflare.com']


### C. Streams and files

When you deal with the big files you should take care about the RAM. Today 1GB won't suprise anyone on the desktop, but server machines, which implement crawlers, may be optimized for the resource.

Using streams instead of RAM-cached files is a good strategy.

- Look for solution here: https://stackoverflow.com/a/16696317
- Look for the sample big file here: http://xcal1.vodafone.co.uk/
- Read about python memory measurement here: https://pythonspeed.com/articles/measuring-memory-python/

In [5]:
import psutil, gc 

def get_mem():
    return psutil.Process().memory_info().rss

In [6]:
large_file_url = "http://212.183.159.230/100MB.zip"

First, download the file as you would do it simple way:

In [7]:
gc.collect()
print("Resident set size:", get_mem())
data = requests.get(large_file_url).content
print("Resident set size:", get_mem())

with open('100-RAM', 'wb') as f:
    f.write(data)

print("Resident set size:", get_mem())
data = None
gc.collect()
print("Resident set size:", get_mem())

Resident set size: 72949760
Resident set size: 177684480
Resident set size: 177684480
Resident set size: 72921088


And then use the streaming mode of the `requests` library.

In [8]:
import requests
import shutil

def download_file(url, destination):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(destination, 'wb') as f:
            shutil.copyfileobj(r.raw, f)

gc.collect()
print("Resident set size:", get_mem())
download_file(large_file_url, "100-stream")
print("Resident set size:", get_mem())

Resident set size: 72921088
Resident set size: 72921088


### D. BeautifulSoup

Plain text HTML is a mixture of content, markup, and code. Extracting structure, or URLs, or plain text might be tricky with regular expressions. 

Building a DOM model is slow, but may save a lot of code and keep you from mistakes.

#### a. Extract all sentences
For indexing and semantic analysis we use different granularity. Often sentence is a good choice. 

In [9]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/firasj/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [10]:
from bs4 import BeautifulSoup
from nltk import tokenize

doc_url = "https://math.stackexchange.com/questions/"\
        "411486/understanding-the-singular-value-decomposition-svd"

text = requests.get(doc_url).text
dom = BeautifulSoup(text)
paragraphs = [p.strip() for p in dom.text.split('\n') if p.strip()]
print(paragraphs)
sents = []
for p in paragraphs:
    sents += tokenize.sent_tokenize(p)

print(sents[90:100])

['Just a moment...Enable JavaScript and cookies to continue']
[]


#### b. Extract URLs from nodes

Be careful with relative links. How would you process them?

In [11]:
import urllib.parse

all_hrefs = dom.find_all('a', href=True)
all_urls = set()
for a in all_hrefs:
    url = a['href']
    url = urllib.parse.urljoin(doc_url, url)
    all_urls.add(url)

all_urls = list(all_urls)
all_urls[:10]

[]

Discuss the next result:

In [12]:
print("|DOM ∩ REGX| =", len(set(all_urls) & set(regexp_urls)))
print("|DOM \ REGX| =", len(set(all_urls) - set(regexp_urls)))
print("|REGX \ DOM| =", len(set(regexp_urls) - set(all_urls)))

|DOM ∩ REGX| = 0
|DOM \ REGX| = 0
|REGX \ DOM| = 1


# 2. Parsing different formats

Most probably, if you meet something in the Internet, this is one of: binary, plain text, XML, or json. XML then splits into xHTML, RSS, Atom, SOAP, XML-RPC, ... . Your task is to learn, how to process different formats.


## 2.1. JSON

In [the given file](https://filesamples.com/samples/code/json/sample4.json) there is valid json. Parse this file and print all people who have `females` gender. Use built-in features of `requests`, or just a `json` library ([ref](https://docs.python.org/3/library/json.html)).

Hint:
- if the file has issues with parsing read about [the difference](https://stackoverflow.com/questions/57152985/what-is-the-difference-between-utf-8-and-utf-8-sig).

In [ ]:
import json
import requests

url = "https://filesamples.com/samples/code/json/sample4.json"

with requests.get(url) as response:
    raw_bytes = response.content
    response.raise_for_status()

    print(raw_bytes)
    content = str(raw_bytes, encoding="utf-8")
    content = json.loads(raw_bytes)

    print("People with 'female' gender:")
    for item in content['people']:
        # print(item)
        if 'female' in item['gender']:
            print(f"{item['firstName']} {item['lastName']} at {item['age']}")

b'{\n  "people" : [\n    {\n       "firstName": "Joe",\n       "lastName": "Jackson",\n       "gender": "male",\n       "age": 28,\n       "number": "7349282382"\n    },\n    {\n       "firstName": "James",\n       "lastName": "Smith",\n       "gender": "male",\n       "age": 32,\n       "number": "5678568567"\n    },\n    {\n       "firstName": "Emily",\n       "lastName": "Jones",\n       "gender": "female",\n       "age": 24,\n       "number": "456754675"\n    }\n  ]\n}'
People with 'female' gender:
Emily Jones at 24


In [24]:
text = "Hello, 🙂_🐍!"
encoded_bytes = text.encode("utf-8")
print(encoded_bytes, type(encoded_bytes))  # Output: b'Hello, \xf0\x9f\x90\x8d!'
decoded_text = encoded_bytes.decode("utf-8")
print(decoded_text, type(decoded_text))


b'Hello, \xf0\x9f\x99\x82_\xf0\x9f\x90\x8d!' <class 'bytes'>
Hello, 🙂_🐍! <class 'str'>


In [25]:
# Wikipedia raw article data containing multiple international characters
url = "https://eduwiki.innopolis.university/index.php/BSc:_Information_Retrieval"

# Fetching the raw bytes from the internet
with requests.get(url) as response:
    raw_bytes = response.content

    # SUCCESS: Properly decoding as UTF-8
    text = raw_bytes.decode("utf-8")
    print("UTF-8 decoded preview:", text[:100])

    # ERROR DEMO: Forcing ASCII will throw a UnicodeDecodeError due to the special characters
    try:
        raw_bytes.decode("ascii")
    except UnicodeDecodeError as e:
        print(f"\nCaught expected error: {e}")

UTF-8 decoded preview: <!DOCTYPE html>
<html class="client-nojs" lang="en" dir="ltr">
<head>
<meta charset="UTF-8"/>
<title

Caught expected error: 'ascii' codec can't decode byte 0xe2 in position 10223: ordinal not in range(128)


In [22]:
# A classic book file explicitly encoded by Gutenberg in ISO-8859-1
url = "https://gutenberg.org" 

with requests.get(url) as response:
    raw_bytes = response.content

    # SUCCESS: Decode using the file's native format
    text = raw_bytes.decode("iso-8859-1")
    print("Latin-1 decoded preview:", text[1000:1100])
    

Latin-1 decoded preview:     content="615269807" >
 <meta property="fb:app_id"       content="115319388529183" >
 <meta prope


[ASCII](https://en.wikipedia.org/wiki/ASCII) encodes each code-point as a value from 0 to 127 – storable as a seven-bit intege. Ninety-five code-points are printable, including digits 0 to 9, lowercase letters a to z, uppercase letters A to Z, and commonly used punctuation symbols. For example, the letter `i` is represented as 105 (decimal).

**ASCII** was the first character encoding standard for the web. It defined 128 different latin characters that could be used on the internet:
* English letters (a-z and A-Z)
* Numbers (0-9)
* Some special characters: ! $ + - ( ) @ < > . # ?


**ANSI (Windows-1252)** was the first Windows character set:
* Identical to ASCII for the first 127 characters
* Special characters from 128 to 159
* Identical to UTF-8 from 160 to 255

The **ISO-8859-1** Character Set was the default character set for HTML 4.
It supported 256 characters:
* Identical to ASCII for the first 127 characters
* Does not use the characters from 128 to 159
* Identical to ANSI and UTF-8 from 160 to 255


The **UTF-8** Character Set
* Identical to ASCII for the values from 0 to 127
* Does not use the characters from 128 to 159
* Identical to ANSI and 8859-1 from 160 to 255
* Continues from the value 256 to 10 000 characters

[Percent-encoding](https://en.wikipedia.org/wiki/Percent-encoding), also known as URL encoding, is a method to encode arbitrary data in a uniform resource identifier (URI) using only the ASCII characters legal within a URI. Percent-encoding is used to ensure special characters do not interfere with the URI's structure and interpretation. Special characters are replaced with a percent sign (%) followed by two hexadecimal digits representing the character's byte value. For example, a space is commonly encoded as `%20`:
* original: `http://example.com/my file.txt`
* encoded: `http://example.com/my%20file.txt`

## 2.2. HTML

For a given StackExchange answer extract logins of the contributors (who asked and who answered) with votes. [bs4](https://beautiful-soup-4.readthedocs.io/en/latest/) will help you to do the job.

I can recommend to use CSS or XPath selectors. `div` elements with `post-layout` class represent answers. Inside there are `div` with `votecell` class stroring votes number and `div` with class `user-details` storing user info. My personal recommendation is to use `css selectors`, which are [documented here](https://beautiful-soup-4.readthedocs.io/en/latest/#css-selectors).

In [26]:
import requests
from bs4 import BeautifulSoup

url = "https://math.stackexchange.com/questions/411486/"\
        "understanding-the-singular-value-decomposition-svd"

# url = "https://stackoverflow.com/questions/7708368/how-can-i-convert-an-image-to-grayscale-via-the-command-line"

print(url)

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.content, 'html.parser')

# Find all posts (question and answers)
# Each post has class 'post-layout' or is a question with 'question' class
posts = soup.select('.question, .answer')

print("Contributors and their votes:")
for post in posts:
    # Get votes
    vote_cell = post.select_one('.votecell .js-vote-count')
    votes = vote_cell.text.strip() if vote_cell else "0"
    
    # Get user info
    user_details = post.select_one('.user-details')
    if user_details:
        user_link = user_details.select_one('a')
        username = user_link.text.strip() if user_link else "Unknown"
    else:
        # Check if it's the question author
        question_user = post.select_one('.question .user-details a')
        username = question_user.text.strip() if question_user else "Unknown"
    
    print(f"{votes} {username}")

response.close()

https://math.stackexchange.com/questions/411486/understanding-the-singular-value-decomposition-svd


HTTPError: 403 Client Error: Forbidden for url: https://math.stackexchange.com/questions/411486/understanding-the-singular-value-decomposition-svd

In [28]:
url = "https://eduwiki.innopolis.university/index.php/BSc:_Information_Retrieval"
# url = "https://eduwiki.innopolis.university/index.php/BSc:_Operating_Systems"

print(url)

with requests.get(url) as response:
    raw_bytes = response.content
    response.raise_for_status()

    soup = BeautifulSoup(raw_bytes, 'html.parser')

    # Find all posts (question and answers)
    # Each post has class 'post-layout' or is a question with 'question' class
    sections_table = soup.select('.wikitable')

    print("Course Sections:")
    for section in sections_table:
        caption = section.select_one('caption')
        if caption:
            if "Topics" in caption.text:
                for row in section.select("tr"):
                    if row.select_one("td"):
                        print(row.select("td")[0].text.strip())

https://eduwiki.innopolis.university/index.php/BSc:_Information_Retrieval
Course Sections:
Information retrieval basics
Text processing and indexing
Vector model and vector indexing
Advanced topics. Media processing


# 2.3. RSS feed

A lot of information is already organized in typed XML documents. Podcasts, for example, are just RSS feed. Parse some rss feed from [rt.com](https://www.rt.com/rss-feeds/) and print out:
- the number of news entries
- the length of the time span between the first and the last news (in days).

Use [`feedparser` library for this](https://waylonwalker.com/parsing-rss-python/).

In [30]:
import feedparser
from datetime import datetime

rss = "https://www.rt.com/rss/pop-culture"

feed = feedparser.parse(rss)
print(feed.entries)
# Number of episodes
count = len(feed.entries)
print(f"Number of news items: {count}")

if count > 0:
    # Get published dates
    dates = []
    for entry in feed.entries:
        if 'published_parsed' in entry:
            dates.append(datetime(*entry.published_parsed[:6]))
    
    if dates:
        first_date = min(dates)
        last_date = max(dates)
        time_span = (last_date - first_date).days
        print(f"Time span between first and last news: {time_span} days")
        # the following compuatation is not very precise!!!! 
        # (as it does not account particular days and years)
        # https://stackoverflow.com/a/4040338
        days = time_span
        years = days//365
        months = (days-365*years)//30
        days = days - years*365 - months*30
        print("News time length is {} months, {} days".format(months, days))
    else:
        print("No publication dates found")
else:
    print("No news found")

[{'title': 'Netherlands dumps Eurovision over Israel', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.rt.com/rss/pop-culture/', 'value': 'Netherlands dumps Eurovision over Israel'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.rt.com/pop-culture/644607-netherlands-dump-eurovision-israel/?utm_source=rss&utm_medium=rss&utm_campaign=RSS'}, {'type': 'image/jpeg', 'length': '123', 'href': 'https://mf.b37mrtl.ru/files/2026.08/thumbnail/6a8cc15d20302735314f7453.jpg', 'rel': 'enclosure'}], 'link': 'https://www.rt.com/pop-culture/644607-netherlands-dump-eurovision-israel/?utm_source=rss&utm_medium=rss&utm_campaign=RSS', 'id': 'https://www.rt.com/pop-culture/644607-netherlands-dump-eurovision-israel/', 'guidislink': False, 'summary': '<img align="left" alt="Preview" src="https://mf.b37mrtl.ru/files/2026.08/thumbnail/6a8cc15d20302735314f7453.jpg" style="margin-right: 10px;" /> Dutch public broadcaster Avrotros has announced it will no lon

# 3. [EXTRA TASK] Solving simple information retrieval task

According to the name, `information retrieval` is the discipline, which helps retrieves information (from unstructured sources). Thus, we will retrieve some information from [this news article](https://www.bbc.com/news/world-us-canada-59944889). Your task is to write a code, which will answer the question: **How many people die every day in the US waiting for a transplant?** Write flexible enough code. Test yourself by changing the link to [this one](https://www.americantransplantfoundation.org/about-transplant/facts-and-myths/).

In [ ]:
import requests
url = 'https://www.bbc.com/news/world-us-canada-59944889'
url2 = 'https://www.americantransplantfoundation.org/about-transplant/facts-and-myths/'

question = 'How many people die every day in the US waiting for a transplant?'

resp = requests.get(url=url)
soup = resp.text
soup = BeautifulSoup(soup)
mydivs = soup.find_all("body")[0].text.replace("\n", " ").replace("\r", "")
text = ""
for char in mydivs:
    text+=char
text = text.split('.')
bow = question.split()
scores = []
for sentence in text:
    score = 0
    for word in bow:
        if word in sentence: score+=1
    scores.append(score)
index = scores.index(max(scores))
print(text[index])